# Mind Map & Notes Release Workflow Test

## Overview
Testing mind map creation, note linking, and release workflow across two tenants.

## Service Ports
| Service | Port |
|---------|------|
| Role-Permission | 8080 |
| Auth Service | 8081 |
| Tenant Service | 8082 |
| User Profile | 8083 |
| API Gateway | 8084 |
| MindMap Service | 8087 |
| Notes Service | 8088 |

---

## Scenario

### Tenants (Schools)
| School | Tenant Key |
|--------|------------|
| Euroschool | euroschool |
| Ravishankar School | ravishankarschool |

### Users
| Name | Role | School | Class |
|------|------|--------|-------|
| Amogh | Student | Euroschool | 4-C |
| Priya | Teacher (Science) | Euroschool | 4-C |
| Daksh | Student | Ravishankar School | 4-C |
| Ravi | Teacher (Science) | Ravishankar School | 4-C |

### Mind Maps & Notes
Each teacher creates:
1. **Photosynthesis** mind map with notes - RELEASED to class
2. **Animal Kingdom** mind map with notes - NOT RELEASED (draft)

### Key Design Decisions
- Mind maps are **per teacher** (not shared)
- Release happens at **class level**
- One mind map can contain **multiple notes**
- Notes linked via **tags** (flexible hierarchy)
- Releasing mind map **auto-releases** linked notes
- **Strict tenant isolation** - no cross-school visibility

---

## Setup: Imports and Configuration

In [ ]:
import requests
import json
import uuid

# Service URLs
ROLE_PERMISSION_URL = "http://localhost:8080"
AUTH_URL = "http://localhost:8081"
TENANT_URL = "http://localhost:8082"
USER_PROFILE_URL = "http://localhost:8083"
API_GATEWAY_URL = "http://localhost:8084"
MINDMAP_URL = "http://localhost:8087"
NOTES_URL = "http://localhost:8088"

# Store created resources for cleanup
created_resources = {
    "tenants": [],
    "users": [],
    "roles": [],
    "notes": [],
    "mindmaps": []
}

def print_response(response, label="Response"):
    """Pretty print API response"""
    print(f"{label} - Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except:
        print(response.text)

def get_headers(token=None, tenant_id=None):
    """Build request headers"""
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    if tenant_id:
        headers["X-Tenant-Id"] = str(tenant_id)
    return headers

print("Setup complete!")

---

## Step 1: Get or Create Tenants

In [ ]:
# Get or create Euroschool tenant
print("Checking for Euroschool tenant...")
response = requests.get(f"{TENANT_URL}/v1/tenants")
euroschool_id = None

if response.status_code == 200:
    for t in response.json():
        if t.get("tenantKey") == "euroschool":
            euroschool_id = t.get("id")
            print(f"Euroschool exists (ID: {euroschool_id})")
            break

if not euroschool_id:
    response = requests.post(
        f"{TENANT_URL}/v1/tenants",
        headers={"Content-Type": "application/json"},
        json={"tenantKey": "euroschool", "name": "Euroschool"}
    )
    if response.status_code in [200, 201]:
        euroschool_id = response.json().get("id")
        created_resources["tenants"].append(euroschool_id)
        print(f"Created Euroschool (ID: {euroschool_id})")

print(f"\nEuroschool ID: {euroschool_id}")

In [ ]:
# Get or create Ravishankar School tenant
print("Checking for Ravishankar School tenant...")
response = requests.get(f"{TENANT_URL}/v1/tenants")
ravishankar_id = None

if response.status_code == 200:
    for t in response.json():
        if t.get("tenantKey") == "ravishankarschool":
            ravishankar_id = t.get("id")
            print(f"Ravishankar School exists (ID: {ravishankar_id})")
            break

if not ravishankar_id:
    response = requests.post(
        f"{TENANT_URL}/v1/tenants",
        headers={"Content-Type": "application/json"},
        json={"tenantKey": "ravishankarschool", "name": "Ravishankar School"}
    )
    if response.status_code in [200, 201]:
        ravishankar_id = response.json().get("id")
        created_resources["tenants"].append(ravishankar_id)
        print(f"Created Ravishankar School (ID: {ravishankar_id})")

print(f"\nRavishankar School ID: {ravishankar_id}")

---

## Step 2: Create Users

In [ ]:
# Create Euroschool users: Amogh (student) and Priya (teacher)
print("Creating Euroschool users...")
print("-" * 40)

euroschool_users = [
    {"email": "amogh@euroschool.com", "password": "amogh123", "name": "Amogh", "role": "STUDENT"},
    {"email": "priya@euroschool.com", "password": "priya123", "name": "Priya", "role": "SUBJECT_TEACHER"},
]

euroschool_user_data = {}

for user in euroschool_users:
    response = requests.post(
        f"{AUTH_URL}/auth/signup",
        headers={"Content-Type": "application/json"},
        json={
            "tenantId": str(euroschool_id),
            "email": user["email"],
            "password": user["password"],
            "name": user["name"],
            "joinMethod": "SELF_SIGNUP"
        }
    )
    
    if response.status_code in [200, 201]:
        data = response.json()
        euroschool_user_data[user["name"]] = {
            "userId": data.get("userId"),
            "accessToken": data.get("accessToken"),
            "role": user["role"]
        }
        created_resources["users"].append({"tenant_id": euroschool_id, "user_id": data.get("userId")})
        print(f"{user['name']} ({user['role']}) - Created (ID: {data.get('userId')})")
    else:
        print(f"{user['name']} - Error: {response.text}")

print("-" * 40)
print(f"Euroschool Users: {json.dumps(euroschool_user_data, indent=2)}")

In [ ]:
# Create Ravishankar School users: Daksh (student) and Ravi (teacher)
print("Creating Ravishankar School users...")
print("-" * 40)

ravishankar_users = [
    {"email": "daksh@ravishankar.com", "password": "daksh123", "name": "Daksh", "role": "STUDENT"},
    {"email": "ravi@ravishankar.com", "password": "ravi123", "name": "Ravi", "role": "SUBJECT_TEACHER"},
]

ravishankar_user_data = {}

for user in ravishankar_users:
    response = requests.post(
        f"{AUTH_URL}/auth/signup",
        headers={"Content-Type": "application/json"},
        json={
            "tenantId": str(ravishankar_id),
            "email": user["email"],
            "password": user["password"],
            "name": user["name"],
            "joinMethod": "SELF_SIGNUP"
        }
    )
    
    if response.status_code in [200, 201]:
        data = response.json()
        ravishankar_user_data[user["name"]] = {
            "userId": data.get("userId"),
            "accessToken": data.get("accessToken"),
            "role": user["role"]
        }
        created_resources["users"].append({"tenant_id": ravishankar_id, "user_id": data.get("userId")})
        print(f"{user['name']} ({user['role']}) - Created (ID: {data.get('userId')})")
    else:
        print(f"{user['name']} - Error: {response.text}")

print("-" * 40)
print(f"Ravishankar Users: {json.dumps(ravishankar_user_data, indent=2)}")

---

## Step 3: Setup Roles and Permissions

Assuming roles (STUDENT, SUBJECT_TEACHER) and permissions (NOTE:*, MINDMAP:*) exist from previous setup.
If not, refer to `school_scenario_test.ipynb` for role/permission creation.

In [ ]:
# Verify roles exist for both tenants
print("Verifying roles...")
print("-" * 40)

for tenant_name, tenant_id in [("Euroschool", euroschool_id), ("Ravishankar", ravishankar_id)]:
    response = requests.get(f"{ROLE_PERMISSION_URL}/tenants/{tenant_id}/roles")
    if response.status_code == 200:
        roles = response.json()
        role_names = [r.get("name") for r in roles]
        print(f"{tenant_name}: {role_names}")
    else:
        print(f"{tenant_name}: Error fetching roles")

print("-" * 40)

In [ ]:
# Assign roles to Euroschool users
print("Assigning roles to Euroschool users...")
print("-" * 40)

# Get role IDs for Euroschool
response = requests.get(f"{ROLE_PERMISSION_URL}/tenants/{euroschool_id}/roles")
euroschool_roles = {r.get("name"): r.get("id") for r in response.json()} if response.status_code == 200 else {}

for user_name, user_info in euroschool_user_data.items():
    role_name = user_info["role"]
    role_id = euroschool_roles.get(role_name)
    
    if not role_id:
        print(f"{user_name} - Role {role_name} not found")
        continue
    
    payload = {
        "roleId": role_id,
        "scopeType": "CLASS",
        "scopeId": "4-C",
        "status": "ACTIVE"
    }
    
    response = requests.post(
        f"{ROLE_PERMISSION_URL}/tenants/{euroschool_id}/users/{user_info['userId']}/roles",
        headers={"Content-Type": "application/json"},
        json=payload
    )
    print(f"{user_name} -> {role_name} (4-C) - Status: {response.status_code}")

print("-" * 40)

In [ ]:
# Assign roles to Ravishankar users
print("Assigning roles to Ravishankar users...")
print("-" * 40)

# Get role IDs for Ravishankar
response = requests.get(f"{ROLE_PERMISSION_URL}/tenants/{ravishankar_id}/roles")
ravishankar_roles = {r.get("name"): r.get("id") for r in response.json()} if response.status_code == 200 else {}

for user_name, user_info in ravishankar_user_data.items():
    role_name = user_info["role"]
    role_id = ravishankar_roles.get(role_name)
    
    if not role_id:
        print(f"{user_name} - Role {role_name} not found")
        continue
    
    payload = {
        "roleId": role_id,
        "scopeType": "CLASS",
        "scopeId": "4-C",
        "status": "ACTIVE"
    }
    
    response = requests.post(
        f"{ROLE_PERMISSION_URL}/tenants/{ravishankar_id}/users/{user_info['userId']}/roles",
        headers={"Content-Type": "application/json"},
        json=payload
    )
    print(f"{user_name} -> {role_name} (4-C) - Status: {response.status_code}")

print("-" * 40)

---

## Step 4: Create Notes

Each teacher creates:
1. Photosynthesis notes (will be released)
2. Animal Kingdom notes (will remain draft)

In [ ]:
# Priya (Euroschool) creates notes
priya_token = euroschool_user_data.get("Priya", {}).get("accessToken")
priya_id = euroschool_user_data.get("Priya", {}).get("userId")

print("Priya creating notes for Euroschool...")
print("-" * 40)

priya_notes = [
    {
        "title": "Photosynthesis - Introduction",
        "summary": "How plants make their food using sunlight",
        "contentMd": "# Photosynthesis\n\nPhotosynthesis is the process by which plants convert sunlight into food.\n\n## Key Components\n- Sunlight\n- Water\n- Carbon Dioxide\n- Chlorophyll\n\n## Formula\n$$6CO_2 + 6H_2O \\xrightarrow{light} C_6H_{12}O_6 + 6O_2$$",
        "tags": ["science", "biology", "plants", "photosynthesis"],
        "classId": "4-C",
        "subject": "Science"
    },
    {
        "title": "Photosynthesis - Light Reactions",
        "summary": "Understanding the light-dependent reactions",
        "contentMd": "# Light Reactions\n\nLight reactions occur in the thylakoid membrane.\n\n## Steps\n1. Light absorption by chlorophyll\n2. Water splitting\n3. Electron transport chain\n4. ATP and NADPH production",
        "tags": ["science", "biology", "photosynthesis", "light-reactions"],
        "classId": "4-C",
        "subject": "Science"
    },
    {
        "title": "Animal Kingdom - Overview",
        "summary": "Classification of animals (DRAFT - not released)",
        "contentMd": "# Animal Kingdom\n\n**DRAFT - Work in Progress**\n\n## Classification\n- Vertebrates\n- Invertebrates\n\n*More content coming soon...*",
        "tags": ["science", "biology", "animals", "animal-kingdom"],
        "classId": "4-C",
        "subject": "Science"
    }
]

euroschool_notes = {}

for note in priya_notes:
    response = requests.post(
        f"{NOTES_URL}/notes",
        headers=get_headers(priya_token, euroschool_id),
        json=note
    )
    
    if response.status_code in [200, 201]:
        data = response.json()
        note_id = data.get("id")
        euroschool_notes[note["title"]] = {
            "id": note_id,
            "status": data.get("status"),
            "tags": note["tags"]
        }
        created_resources["notes"].append({"tenant_id": euroschool_id, "note_id": note_id})
        print(f"Created: {note['title']} (ID: {note_id})")
    else:
        print(f"Failed: {note['title']} - {response.text}")

print("-" * 40)
print(f"Euroschool Notes: {json.dumps(euroschool_notes, indent=2)}")

In [ ]:
# Ravi (Ravishankar) creates notes
ravi_token = ravishankar_user_data.get("Ravi", {}).get("accessToken")
ravi_id = ravishankar_user_data.get("Ravi", {}).get("userId")

print("Ravi creating notes for Ravishankar School...")
print("-" * 40)

ravi_notes = [
    {
        "title": "Photosynthesis - Basics",
        "summary": "Introduction to photosynthesis for beginners",
        "contentMd": "# Photosynthesis Basics\n\nPlants are amazing! They make their own food.\n\n## What plants need\n- Sunshine\n- Water from roots\n- Air (CO2)\n\n## What plants make\n- Food (glucose)\n- Oxygen (for us to breathe!)",
        "tags": ["science", "biology", "plants", "photosynthesis"],
        "classId": "4-C",
        "subject": "Science"
    },
    {
        "title": "Animal Kingdom - Introduction",
        "summary": "Types of animals (DRAFT - not released)",
        "contentMd": "# Animal Kingdom\n\n**DRAFT**\n\n## Types of Animals\n- Mammals\n- Birds\n- Reptiles\n- Fish\n- Insects\n\n*Still preparing this content...*",
        "tags": ["science", "biology", "animals", "animal-kingdom"],
        "classId": "4-C",
        "subject": "Science"
    }
]

ravishankar_notes = {}

for note in ravi_notes:
    response = requests.post(
        f"{NOTES_URL}/notes",
        headers=get_headers(ravi_token, ravishankar_id),
        json=note
    )
    
    if response.status_code in [200, 201]:
        data = response.json()
        note_id = data.get("id")
        ravishankar_notes[note["title"]] = {
            "id": note_id,
            "status": data.get("status"),
            "tags": note["tags"]
        }
        created_resources["notes"].append({"tenant_id": ravishankar_id, "note_id": note_id})
        print(f"Created: {note['title']} (ID: {note_id})")
    else:
        print(f"Failed: {note['title']} - {response.text}")

print("-" * 40)
print(f"Ravishankar Notes: {json.dumps(ravishankar_notes, indent=2)}")

---

## Step 5: Create Mind Maps

Each teacher creates:
1. Photosynthesis mind map (will be released)
2. Animal Kingdom mind map (will remain draft)

In [ ]:
# Priya creates mind maps for Euroschool
print("Priya creating mind maps for Euroschool...")
print("-" * 40)

euroschool_mindmaps = {}

# Photosynthesis Mind Map
response = requests.post(
    f"{MINDMAP_URL}/mindmaps",
    headers=get_headers(priya_token, euroschool_id),
    json={
        "title": "Photosynthesis",
        "subject": "Science",
        "grade": "4",
        "tags": ["biology", "plants", "topic:photosynthesis"],
        "visibility": "TENANT"
    }
)

if response.status_code in [200, 201]:
    data = response.json()
    euroschool_mindmaps["Photosynthesis"] = data.get("mindMapId") or data.get("id")
    created_resources["mindmaps"].append({"tenant_id": euroschool_id, "mindmap_id": euroschool_mindmaps["Photosynthesis"]})
    print(f"Created: Photosynthesis (ID: {euroschool_mindmaps['Photosynthesis']})")
else:
    print(f"Failed: Photosynthesis - {response.text}")

# Animal Kingdom Mind Map
response = requests.post(
    f"{MINDMAP_URL}/mindmaps",
    headers=get_headers(priya_token, euroschool_id),
    json={
        "title": "Animal Kingdom",
        "subject": "Science",
        "grade": "4",
        "tags": ["biology", "animals", "topic:animal-kingdom"],
        "visibility": "TENANT"
    }
)

if response.status_code in [200, 201]:
    data = response.json()
    euroschool_mindmaps["Animal Kingdom"] = data.get("mindMapId") or data.get("id")
    created_resources["mindmaps"].append({"tenant_id": euroschool_id, "mindmap_id": euroschool_mindmaps["Animal Kingdom"]})
    print(f"Created: Animal Kingdom (ID: {euroschool_mindmaps['Animal Kingdom']})")
else:
    print(f"Failed: Animal Kingdom - {response.text}")

print("-" * 40)
print(f"Euroschool Mind Maps: {euroschool_mindmaps}")

In [ ]:
# Ravi creates mind maps for Ravishankar School
print("Ravi creating mind maps for Ravishankar School...")
print("-" * 40)

ravishankar_mindmaps = {}

# Photosynthesis Mind Map
response = requests.post(
    f"{MINDMAP_URL}/mindmaps",
    headers=get_headers(ravi_token, ravishankar_id),
    json={
        "title": "Photosynthesis",
        "subject": "Science",
        "grade": "4",
        "tags": ["biology", "plants", "topic:photosynthesis"],
        "visibility": "TENANT"
    }
)

if response.status_code in [200, 201]:
    data = response.json()
    ravishankar_mindmaps["Photosynthesis"] = data.get("mindMapId") or data.get("id")
    created_resources["mindmaps"].append({"tenant_id": ravishankar_id, "mindmap_id": ravishankar_mindmaps["Photosynthesis"]})
    print(f"Created: Photosynthesis (ID: {ravishankar_mindmaps['Photosynthesis']})")
else:
    print(f"Failed: Photosynthesis - {response.text}")

# Animal Kingdom Mind Map
response = requests.post(
    f"{MINDMAP_URL}/mindmaps",
    headers=get_headers(ravi_token, ravishankar_id),
    json={
        "title": "Animal Kingdom",
        "subject": "Science",
        "grade": "4",
        "tags": ["biology", "animals", "topic:animal-kingdom"],
        "visibility": "TENANT"
    }
)

if response.status_code in [200, 201]:
    data = response.json()
    ravishankar_mindmaps["Animal Kingdom"] = data.get("mindMapId") or data.get("id")
    created_resources["mindmaps"].append({"tenant_id": ravishankar_id, "mindmap_id": ravishankar_mindmaps["Animal Kingdom"]})
    print(f"Created: Animal Kingdom (ID: {ravishankar_mindmaps['Animal Kingdom']})")
else:
    print(f"Failed: Animal Kingdom - {response.text}")

print("-" * 40)
print(f"Ravishankar Mind Maps: {ravishankar_mindmaps}")

---

## Step 6: Add Nodes to Mind Maps (Link Notes)

In [ ]:
# Add nodes to Euroschool Photosynthesis mind map
print("Adding nodes to Euroschool Photosynthesis mind map...")
print("-" * 40)

photosynthesis_map_id = euroschool_mindmaps.get("Photosynthesis")

if photosynthesis_map_id:
    # Root node
    root_node_id = str(uuid.uuid4())
    response = requests.put(
        f"{MINDMAP_URL}/mindmaps/{photosynthesis_map_id}/nodes/{root_node_id}",
        headers=get_headers(priya_token, euroschool_id),
        json={
            "type": "CONCEPT",
            "title": "Photosynthesis",
            "posX": 400,
            "posY": 100
        }
    )
    print(f"Root node - Status: {response.status_code}")
    
    # Introduction note node
    intro_note = euroschool_notes.get("Photosynthesis - Introduction", {})
    if intro_note.get("id"):
        intro_node_id = str(uuid.uuid4())
        response = requests.put(
            f"{MINDMAP_URL}/mindmaps/{photosynthesis_map_id}/nodes/{intro_node_id}",
            headers=get_headers(priya_token, euroschool_id),
            json={
                "type": "NOTE_REF",
                "title": "Introduction",
                "contentId": intro_note["id"],
                "posX": 200,
                "posY": 250
            }
        )
        print(f"Introduction node (linked to note) - Status: {response.status_code}")
    
    # Light Reactions note node
    light_note = euroschool_notes.get("Photosynthesis - Light Reactions", {})
    if light_note.get("id"):
        light_node_id = str(uuid.uuid4())
        response = requests.put(
            f"{MINDMAP_URL}/mindmaps/{photosynthesis_map_id}/nodes/{light_node_id}",
            headers=get_headers(priya_token, euroschool_id),
            json={
                "type": "NOTE_REF",
                "title": "Light Reactions",
                "contentId": light_note["id"],
                "posX": 600,
                "posY": 250
            }
        )
        print(f"Light Reactions node (linked to note) - Status: {response.status_code}")

print("-" * 40)

In [ ]:
# Add nodes to Euroschool Animal Kingdom mind map (draft)
print("Adding nodes to Euroschool Animal Kingdom mind map (draft)...")
print("-" * 40)

animal_map_id = euroschool_mindmaps.get("Animal Kingdom")

if animal_map_id:
    # Root node
    root_node_id = str(uuid.uuid4())
    response = requests.put(
        f"{MINDMAP_URL}/mindmaps/{animal_map_id}/nodes/{root_node_id}",
        headers=get_headers(priya_token, euroschool_id),
        json={
            "type": "CONCEPT",
            "title": "Animal Kingdom",
            "posX": 400,
            "posY": 100
        }
    )
    print(f"Root node - Status: {response.status_code}")
    
    # Overview note node
    overview_note = euroschool_notes.get("Animal Kingdom - Overview", {})
    if overview_note.get("id"):
        overview_node_id = str(uuid.uuid4())
        response = requests.put(
            f"{MINDMAP_URL}/mindmaps/{animal_map_id}/nodes/{overview_node_id}",
            headers=get_headers(priya_token, euroschool_id),
            json={
                "type": "NOTE_REF",
                "title": "Overview",
                "contentId": overview_note["id"],
                "posX": 400,
                "posY": 250
            }
        )
        print(f"Overview node (linked to note) - Status: {response.status_code}")

print("-" * 40)

---

## Step 7: Release Photosynthesis Mind Map (NOT Animal Kingdom)

Only releasing Photosynthesis to class 4-C. Animal Kingdom remains draft.

In [ ]:
# Priya publishes Photosynthesis mind map to class 4-C
print("Priya publishing Photosynthesis mind map to class 4-C...")
print("-" * 40)

photosynthesis_map_id = euroschool_mindmaps.get("Photosynthesis")

if photosynthesis_map_id:
    response = requests.post(
        f"{MINDMAP_URL}/mindmaps/{photosynthesis_map_id}/publish",
        headers=get_headers(priya_token, euroschool_id),
        json={
            "publishTargets": {
                "scope": "CLASS",
                "classIds": ["4-C"]
            }
        }
    )
    
    print(f"Publish status: {response.status_code}")
    if response.status_code == 200:
        print("[PASS] Photosynthesis mind map published to 4-C!")
        print("Linked notes should now be visible to students.")
    else:
        print(f"Response: {response.text}")

print("-" * 40)
print("NOTE: Animal Kingdom mind map remains as DRAFT (not published)")

In [ ]:
# Ravi publishes Photosynthesis mind map to class 4-C (Ravishankar)
print("Ravi publishing Photosynthesis mind map to class 4-C...")
print("-" * 40)

photosynthesis_map_id = ravishankar_mindmaps.get("Photosynthesis")

if photosynthesis_map_id:
    # First add a node with note reference
    photo_note = ravishankar_notes.get("Photosynthesis - Basics", {})
    if photo_note.get("id"):
        node_id = str(uuid.uuid4())
        requests.put(
            f"{MINDMAP_URL}/mindmaps/{photosynthesis_map_id}/nodes/{node_id}",
            headers=get_headers(ravi_token, ravishankar_id),
            json={
                "type": "NOTE_REF",
                "title": "Basics",
                "contentId": photo_note["id"],
                "posX": 400,
                "posY": 200
            }
        )
    
    # Publish
    response = requests.post(
        f"{MINDMAP_URL}/mindmaps/{photosynthesis_map_id}/publish",
        headers=get_headers(ravi_token, ravishankar_id),
        json={
            "publishTargets": {
                "scope": "CLASS",
                "classIds": ["4-C"]
            }
        }
    )
    
    print(f"Publish status: {response.status_code}")
    if response.status_code == 200:
        print("[PASS] Photosynthesis mind map published to 4-C!")
    else:
        print(f"Response: {response.text}")

print("-" * 40)

---

## Step 8: Test Access Scenarios

In [ ]:
# Test 1: Amogh (Euroschool student) can see PUBLISHED Photosynthesis notes
print("Test 1: Amogh accessing published Photosynthesis notes...")
print("-" * 40)

amogh_token = euroschool_user_data.get("Amogh", {}).get("accessToken")

response = requests.get(
    f"{NOTES_URL}/notes",
    headers=get_headers(amogh_token, euroschool_id),
    params={"tags": "photosynthesis"}
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", []) if isinstance(data, dict) else data
    
    published_notes = [n for n in notes if n.get("status") == "PUBLISHED"]
    print(f"Amogh can see {len(published_notes)} published Photosynthesis note(s):")
    for note in published_notes:
        print(f"  - {note.get('title')}")
    
    if len(published_notes) > 0:
        print("\n[PASS] Amogh can access published notes")
    else:
        print("\n[INFO] No published notes visible yet")
else:
    print(f"Error: {response.text}")

In [ ]:
# Test 2: Amogh CANNOT see DRAFT Animal Kingdom notes
print("Test 2: Amogh trying to access DRAFT Animal Kingdom notes (should not see)...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers=get_headers(amogh_token, euroschool_id),
    params={"tags": "animal-kingdom"}
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", []) if isinstance(data, dict) else data
    
    # Filter for published only (student should not see drafts)
    visible_notes = [n for n in notes if n.get("status") == "PUBLISHED"]
    
    if len(visible_notes) == 0:
        print("Amogh cannot see any Animal Kingdom notes (all drafts)")
        print("\n[PASS] Draft notes are hidden from students!")
    else:
        print(f"[FAIL] Amogh can see {len(visible_notes)} Animal Kingdom notes (should be hidden)")
        for note in visible_notes:
            print(f"  - {note.get('title')} (Status: {note.get('status')})")
else:
    print(f"Error: {response.text}")

In [ ]:
# Test 3: Tenant Isolation - Amogh CANNOT see Ravishankar School's notes
print("Test 3: Amogh trying to access Ravishankar School notes (should fail)...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers=get_headers(amogh_token, ravishankar_id)  # Wrong tenant!
)

if response.status_code in [401, 403]:
    print("Access denied (as expected)")
    print("\n[PASS] Tenant isolation working!")
elif response.status_code == 200:
    data = response.json()
    notes = data.get("items", []) if isinstance(data, dict) else data
    if len(notes) == 0:
        print("Empty result (tenant isolation via empty response)")
        print("\n[PASS] Tenant isolation working!")
    else:
        print(f"[FAIL] Amogh can see {len(notes)} notes from Ravishankar School!")
else:
    print(f"Response: {response.text}")

In [ ]:
# Test 4: Daksh (Ravishankar student) can see their published notes
print("Test 4: Daksh accessing Ravishankar School notes...")
print("-" * 40)

daksh_token = ravishankar_user_data.get("Daksh", {}).get("accessToken")

response = requests.get(
    f"{NOTES_URL}/notes",
    headers=get_headers(daksh_token, ravishankar_id)
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", []) if isinstance(data, dict) else data
    
    print(f"Daksh can see {len(notes)} note(s):")
    for note in notes:
        print(f"  - {note.get('title')} (Status: {note.get('status')})")
    
    print("\n[PASS] Daksh can access his school's notes")
else:
    print(f"Error: {response.text}")

In [ ]:
# Test 5: Teacher (Priya) can see both DRAFT and PUBLISHED notes
print("Test 5: Priya (teacher) accessing all her notes including drafts...")
print("-" * 40)

response = requests.get(
    f"{NOTES_URL}/notes",
    headers=get_headers(priya_token, euroschool_id)
)

if response.status_code == 200:
    data = response.json()
    notes = data.get("items", []) if isinstance(data, dict) else data
    
    draft_count = len([n for n in notes if n.get("status") == "DRAFT"])
    published_count = len([n for n in notes if n.get("status") == "PUBLISHED"])
    
    print(f"Priya can see {len(notes)} total note(s):")
    print(f"  - DRAFT: {draft_count}")
    print(f"  - PUBLISHED: {published_count}")
    
    for note in notes:
        print(f"  - {note.get('title')} (Status: {note.get('status')})")
    
    if draft_count > 0:
        print("\n[PASS] Teacher can see her draft notes")
else:
    print(f"Error: {response.text}")

---

## Step 9: View Mind Maps

In [ ]:
# Amogh views published Photosynthesis mind map
print("Amogh viewing Photosynthesis mind map...")
print("-" * 40)

photosynthesis_map_id = euroschool_mindmaps.get("Photosynthesis")

if photosynthesis_map_id:
    response = requests.get(
        f"{MINDMAP_URL}/mindmaps/{photosynthesis_map_id}/graph",
        headers=get_headers(amogh_token, euroschool_id),
        params={"version": "published"}
    )
    
    if response.status_code == 200:
        graph = response.json()
        nodes = graph.get("nodes", [])
        edges = graph.get("edges", [])
        
        print(f"Mind map has {len(nodes)} node(s) and {len(edges)} edge(s)")
        print("\nNodes:")
        for node in nodes:
            print(f"  - {node.get('title')} (Type: {node.get('type')})")
            if node.get("contentId"):
                print(f"    Linked Note ID: {node.get('contentId')}")
        
        print("\n[PASS] Amogh can view the published mind map")
    else:
        print(f"Cannot access: {response.status_code}")
        print(response.text)

In [ ]:
# Amogh tries to view DRAFT Animal Kingdom mind map (should fail)
print("Amogh trying to view DRAFT Animal Kingdom mind map (should fail)...")
print("-" * 40)

animal_map_id = euroschool_mindmaps.get("Animal Kingdom")

if animal_map_id:
    response = requests.get(
        f"{MINDMAP_URL}/mindmaps/{animal_map_id}/graph",
        headers=get_headers(amogh_token, euroschool_id),
        params={"version": "published"}
    )
    
    if response.status_code in [403, 404]:
        print("Access denied or not found (as expected for unpublished map)")
        print("\n[PASS] Draft mind map is not accessible to students")
    elif response.status_code == 200:
        graph = response.json()
        nodes = graph.get("nodes", [])
        if len(nodes) == 0:
            print("Empty result (draft not published)")
            print("\n[PASS] Draft mind map content hidden from students")
        else:
            print(f"[FAIL] Amogh can see draft mind map with {len(nodes)} nodes!")
    else:
        print(f"Response: {response.status_code} - {response.text}")

---

## Summary

In [ ]:
print("="*60)
print("MIND MAP & NOTES RELEASE WORKFLOW TEST SUMMARY")
print("="*60)
print()
print("Scenario:")
print("-" * 40)
print("Euroschool (4-C):")
print(f"  Teacher: Priya")
print(f"  Student: Amogh")
print(f"  Mind Maps: {list(euroschool_mindmaps.keys())}")
print(f"  Notes: {list(euroschool_notes.keys())}")
print()
print("Ravishankar School (4-C):")
print(f"  Teacher: Ravi")
print(f"  Student: Daksh")
print(f"  Mind Maps: {list(ravishankar_mindmaps.keys())}")
print(f"  Notes: {list(ravishankar_notes.keys())}")
print()
print("Release Status:")
print("-" * 40)
print("  Photosynthesis: RELEASED to class 4-C")
print("  Animal Kingdom: DRAFT (not released)")
print()
print("Key Behaviors Tested:")
print("-" * 40)
print("  1. Students see only PUBLISHED notes")
print("  2. Students cannot see DRAFT notes")
print("  3. Teachers can see their DRAFT notes")
print("  4. Mind map release cascades to linked notes")
print("  5. Strict tenant isolation between schools")
print("="*60)

---

## Cleanup

In [ ]:
# View all created resources
print("Created Resources:")
print(json.dumps(created_resources, indent=2))

In [ ]:
# Optional: Cleanup all created resources
# Uncomment to run

# print("Cleaning up...")
# print("-" * 40)

# # Delete notes
# for note_info in created_resources.get("notes", []):
#     response = requests.delete(
#         f"{NOTES_URL}/notes/{note_info['note_id']}",
#         headers={"X-Tenant-Id": str(note_info['tenant_id'])}
#     )
#     print(f"Deleted note {note_info['note_id']} - Status: {response.status_code}")

# # Delete mindmaps
# for mm_info in created_resources.get("mindmaps", []):
#     response = requests.delete(
#         f"{MINDMAP_URL}/mindmaps/{mm_info['mindmap_id']}",
#         headers={"X-Tenant-Id": str(mm_info['tenant_id'])}
#     )
#     print(f"Deleted mindmap {mm_info['mindmap_id']} - Status: {response.status_code}")

# print("-" * 40)
# print("Cleanup complete!")